Notebook to test the visualisation functions in visualization/candle_plots.py

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.data.load_candle_data import (
    load_candle_splits,
    clean_candle_splits,
    describe_split,
    validate_clean_split,
    compute_log_returns,
    get_channel
)

from src.visualization.candle_plots import (
    plot_return_correlation_heatmap,
    plot_intraday_channel,
    plot_intraday_log_returns,
    plot_average_intraday_abs_return
)

In [ ]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/Shared drives/Vishal/data/cached_datasets/exp-1m-95s-24y/session"
)

DATA_DIR.exists(), DATA_DIR

In [ ]:
train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

describe_split(train_raw, "raw train")
describe_split(val_raw, "raw val")
describe_split(test_raw, "raw test")

In [ ]:
train, val, test = clean_candle_splits(train_raw, val_raw, test_raw)

describe_split(train, "clean train")
describe_split(val, "clean val")
describe_split(test, "clean test")
print("\n")
validate_clean_split(train)
validate_clean_split(val)
validate_clean_split(test)

In [ ]:
x, aux, day = train["samples"][0]

returns = compute_log_returns(x=x,split=train,channels=['close'])

print("day:", day)
print("x shape:", tuple(x.shape))
print("returns shape:", tuple(returns.shape))
print("returns mean:", returns.mean().item())
print("returns std:", returns.std().item())
print("returns has NaN:", torch.isnan(returns).any().item())
print("returns has Inf:", torch.isinf(returns).any().item())

In [ ]:
#check for error data - flag assets where there are large intraday jumps in prices
threshold = 100
flagged_asset_ids = set()

for x, _, day in train["samples"]:
    open_prices = get_channel(x, train, "open")
    close_prices = get_channel(x, train, "close")

    diff = open_prices[1:] - close_prices[:-1]

    asset_has_large_gap = diff.abs().ge(threshold).any(dim=0)

    flagged_ids_this_day = torch.where(asset_has_large_gap)[0].tolist()

    flagged_asset_ids.update(flagged_ids_this_day)

flagged_tickers = [
    train["asset_cols"][asset_id]
    for asset_id in sorted(flagged_asset_ids)
]

flagged_tickers

In [ ]:
#check for errors - flag assets where there are large overnight moves in prices. These will likely be stock splits
threshold = 100
flagged_asset_ids = set()

samples = train["samples"]

for sample_idx in range(1, len(samples)):
    x_prev, _, prev_day = samples[sample_idx - 1]
    x_curr, _, curr_day = samples[sample_idx]

    prev_close = get_channel(x_prev, train, "close")[-1]
    curr_open = get_channel(x_curr, train, "open")[0]

    gap = curr_open - prev_close

    asset_has_large_gap = gap.abs().ge(threshold)

    flagged_ids_this_gap = torch.where(asset_has_large_gap)[0].tolist()

    flagged_asset_ids.update(flagged_ids_this_gap)

flagged_tickers = [
    train["asset_cols"][asset_id]
    for asset_id in sorted(flagged_asset_ids)
]

flagged_tickers

In [ ]:
fig, ax, corr, labels = plot_return_correlation_heatmap(
    split=train,
    channel='close',
    sample_indices=None,
    assets=None,
    reorder=False,
    show_tickers=True,
)

print("corr shape:", corr.shape)
print("num labels:", len(labels))

In [ ]:
fig, ax, corr_clustered, labels_clustered = plot_return_correlation_heatmap(
    split=train,
    channel='close',
    sample_indices=[148],
    assets=None,
    reorder=True,
    cluster_by_abs=True,
    show_tickers=True,
)

print("clustered corr shape:", corr_clustered.shape)
print("first 10 clustered labels:", labels_clustered[:10])

In [ ]:
selected_assets = ["AAPL", "AMD", "AMZN", "MSFT"]

fig, ax, corr_small, labels_small = plot_return_correlation_heatmap(
    split=train,
    channel='close',
    sample_indices=None,
    assets=selected_assets,
    reorder=False,
    show_tickers=True,
)

print("small corr shape:", corr_small.shape)
print("labels:", labels_small)

In [ ]:
fig, ax = plot_intraday_channel(
    split=train,
    channel='close',
    sample_indices=[110],
    assets="NVDA",
    normalize=False,
    mode="concat",
    max_samples=None
)

In [ ]:
fig, ax = plot_intraday_channel(
    split=train,
    channel='volume',
    sample_indices=[100],
    assets=["GOOG"],
    normalize=False,
    mode="overlay",
    max_samples=None
)

In [ ]:
fig, ax = plot_intraday_log_returns(
    split=train,
    channel='close',
    sample_indices=None,
    assets="AAPL",
    mode="concat",
)

In [ ]:
fig, ax = plot_intraday_log_returns(
    split=train,
    channel='open',
    sample_indices=slice(100, 110),
    assets="AAPL",
    mode="concat",
)

In [ ]:
fig, ax, avg_abs_return, std_abs_return = plot_average_intraday_abs_return(
    split=train,
    channel='close',
    sample_indices=None,
    mode="profile",
)

In [ ]:
fig, ax, avg_abs_return, std_abs_return = plot_average_intraday_abs_return(
    split=train,
    channel='close',
    sample_indices=slice(145,155),
    mode="concat",
    assets=None,
)